In [ ]:
import json
import pandas as pd
import networkx as nx
import community as community_louvain # Usa a mesma biblioteca do notebook
import random
from pathlib import Path
from typing import List, Dict, Any, Optional, Set, Tuple
import time 
import sys

# Define um limite de recursão maior para cálculos em grafos complexos
sys.setrecursionlimit(5000)

# --- 1. SEÇÃO DE CONFIGURAÇÃO CENTRALIZADA (COPIADA DA CÉLULA 1) ---
INPUT_DIR = Path('../dados/dados_com_flags_redirecionamento') 
INPUT_FILENAME_WITH_FLAGS = 'dados_api_com_flags_e_refs.json'
REDIRECT_MAP_FILENAME = 'redirect_map.json'
SHORTEST_PATHS_FILENAME = 'shortest_paths_dist.pkl' 

# Parâmetros dos algoritmos
LOUVAIN_RESOLUTION = 0.9
RANDOM_SEED = 42
LAYOUT_SCALE_FACTOR = 8000

print("✅ Configurações e funções do notebook carregadas.\n")

# --- 2. FUNÇÕES AUXILIARES (COPIADAS DA CÉLULA 1) ---

def carregar_dados_json(filepath: Path) -> Optional[List[Dict[str, Any]]]:
    """
    Carrega dados de um arquivo JSON e retorna especificamente a lista
    armazenada sob a chave 'verbetes_completo'.
    """
    if not filepath.exists():
        print(f"ERRO: Arquivo não encontrado em '{filepath}'")
        return None 
    print(f"Carregando dados de '{filepath}'...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            dados_completos = json.load(f)
        lista_verbetes = dados_completos.get('verbetes_completo', [])
        if not isinstance(lista_verbetes, list):
             print(f"ERRO: O conteúdo da chave 'verbetes_completo' em '{filepath}' não é uma lista.")
             return None 
        if not lista_verbetes:
             print(f"AVISO: A lista 'verbetes_completo' em '{filepath}' está vazia.")
        return lista_verbetes 
    except json.JSONDecodeError as e:
        print(f"ERRO: Falha ao decodificar o JSON em '{filepath}': {e}")
        return None
    except Exception as e:
        print(f"Erro inesperado ao carregar JSON em {filepath}: {e}")
        return None

def construir_grafo_com_redirects(verbetes_com_flags: List[Dict[str, Any]],
                                  redirect_map: Dict[str, str],
                                  direcionado=True,
                                  max_redirect_hops=5) -> nx.Graph:
    """
    Constrói grafo resolvendo redirecionamentos, incluindo lógica para
    seguir cadeias de redirects (redirects de redirects).
    """
    
    print(f"Construindo grafo {'direcionado' if direcionado else 'não-direcionado'} com resolução ITERATIVA de redirects...")
    G = nx.DiGraph() if direcionado else nx.Graph()

    # --- 1. Filtrar verbetes reais e criar mapa titulo -> id ---
    verbetes_reais = []
    titulos_ids_reais = {}
    print("  - Filtrando verbetes reais e criando mapa Título->ID...")
    for index, verbete in enumerate(verbetes_com_flags):
         if not isinstance(verbete, dict): continue 
         if not verbete.get('is_redirect') and 'id' in verbete and 'titulo' in verbete:
             verbetes_reais.append(verbete)
             titulos_ids_reais[verbete['titulo']] = str(verbete['id'])
    print(f"  - {len(verbetes_reais)} verbetes reais identificados.")
    if not verbetes_reais: return G

    # --- 2. Adicionar Nós (APENAS verbetes reais) ---
    print("  - Adicionando nós ao grafo...")
    for verbete in verbetes_reais: G.add_node(str(verbete['id']))
    print(f"  - {G.number_of_nodes()} nós adicionados.")

    # --- 3. Adicionar Arestas, com RESOLUÇÃO ITERATIVA ---
    print("  - Adicionando arestas (resolvendo cadeias de redirecionamentos)...")
    arestas_adicionadas = 0
    refs_resolvidas_count = 0
    refs_quebradas_count = 0
    redirect_loops_detected = 0

    DEBUG_LINK_ALVO = "(Des)continuidades na experiência de \"vida sob cerco\" e na \"sociabilidade violenta\" (resenha)"
    DEBUG_VERBETE_ORIGEM = "Favela é comunidade? (artigo)"

    for verbete_origem in verbetes_reais:
        source_vid = str(verbete_origem['id'])
        referencias_originais = verbete_origem.get('referencias', [])
        IS_DEBUG_VERBETE = (verbete_origem['titulo'] == DEBUG_VERBETE_ORIGEM)

        for ref_titulo in referencias_originais:
            if not isinstance(ref_titulo, str) or not ref_titulo: continue

            current_target_title = ref_titulo
            final_target_title = ref_titulo 
            visited_redirects = {ref_titulo} 
            hops = 0 
            IS_DEBUG_LINK = (ref_titulo == DEBUG_LINK_ALVO)
            
            while current_target_title in redirect_map and hops < max_redirect_hops:
                next_target = redirect_map[current_target_title]
                if next_target in visited_redirects:
                    redirect_loops_detected += 1
                    final_target_title = current_target_title 
                    break 
                final_target_title = next_target 
                current_target_title = next_target 
                visited_redirects.add(current_target_title)
                hops += 1
                if hops == 1: 
                    refs_resolvidas_count += 1
            
            if final_target_title in titulos_ids_reais:
                target_vid = titulos_ids_reais[final_target_title]
                if G.has_node(source_vid) and G.has_node(target_vid):
                    G.add_edge(source_vid, target_vid)
                    arestas_adicionadas += 1
            else:
                refs_quebradas_count += 1
                
    # --- 4. Conclusão e Retorno ---
    print(f"Construção do grafo concluída:")
    print(f"  - Nós (Verbetes Reais): {G.number_of_nodes()}")
    print(f"  - Arestas Adicionadas (tentativas): {arestas_adicionadas}")
    print(f"  - Arestas Únicas no Grafo: {G.number_of_edges()}")
    print(f"  - Referências que passaram por resolução (1+ saltos): {refs_resolvidas_count}")
    if redirect_loops_detected > 0:
         print(f"  - Cadeias de redirect interrompidas por loops: {redirect_loops_detected}")
    if refs_quebradas_count > 0:
        print(f"  - Referências ignoradas (alvo final inválido ou loop): {refs_quebradas_count}")
        
    return G


def calcular_metricas_globais_aderentes():
    
    # Dicionário para armazenar todas as métricas
    metricas = {}

    # --- ETAPA 1: Carregar Dados Brutos (Como Célula 2) ---
    print("\n--- ETAPA 1: Carregando Dados Brutos ---")
    input_path = INPUT_DIR / INPUT_FILENAME_WITH_FLAGS
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME

    verbetes_com_flags = carregar_dados_json(input_path)
    if not verbetes_com_flags:
        print("ERRO: Falha ao carregar dados com flags. Abortando.")
        return

    redirect_map = {}
    if redirect_map_path.exists():
        print(f"Carregando mapa de redirecionamentos de '{redirect_map_path}'...")
        with open(redirect_map_path, 'r', encoding='utf-8') as f:
            redirect_map = json.load(f)
        print(f"Mapa carregado com {len(redirect_map)} entradas.")
    else:
        print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado.")

    # --- ETAPA 2: Calcular Métricas Não-Grafo (Categorias, Editores) ---
    print("\n--- ETAPA 2: Calculando Métricas Não-Grafo ---")
    all_categories = set()
    all_editors = set()
    
    for verbete in verbetes_com_flags:
        if not isinstance(verbete, dict) or verbete.get('is_redirect'):
            continue
        
        # Coleta categorias
        for cat in verbete.get('categorias', []):
            if isinstance(cat, str) and cat:
                all_categories.add(cat)
        
        # Coleta editores
        for editor in verbete.get('usuarios_edicoes', {}).keys():
                if isinstance(editor, str) and editor:
                    all_editors.add(editor)

    metricas["Quantidade de Categorias Únicas"] = len(all_categories)
    metricas["Quantidade de Editores Únicos"] = len(all_editors)
    print(f"  - {len(all_categories)} categorias únicas encontradas.")
    print(f"  - {len(all_editors)} editores únicos encontrados.")

    # --- ETAPA 3: Construir Grafos  ---
    print("\n--- ETAPA 3: Construindo Grafos (Metodologia Célula 2) ---")
    
    G_direcionado = construir_grafo_com_redirects(
        verbetes_com_flags, 
        redirect_map, 
        direcionado=True
    )
    
    print("\nCriando versão não-direcionada (G.to_undirected()) para métricas de componente...")
    G_nao_direcionado = G_direcionado.to_undirected()
    print(f"  - Grafo não-direcionado criado com {G_nao_direcionado.number_of_nodes()} nós e {G_nao_direcionado.number_of_edges()} arestas.")

    if G_direcionado.number_of_nodes() == 0:
        print("Grafo resultante está vazio. Encerrando análise.")
        return

    # --- ETAPA 4: Calcular Métricas do Grafo Direcionado ---
    print("\n--- ETAPA 4: Calculando Métricas Globais (Grafo Direcionado) ---")
    
    metricas["Número de Nós (Verbetes)"] = G_direcionado.number_of_nodes()
    metricas["Número de Arestas (Hiperlinks)"] = G_direcionado.number_of_edges()
    metricas["Densidade da Rede"] = nx.density(G_direcionado)
    
    print(f"  - Nós: {metricas['Número de Nós (Verbetes)']}")
    print(f"  - Arestas: {metricas['Número de Arestas (Hiperlinks)']}")
    print(f"  - Densidade: {metricas['Densidade da Rede']:.6f}")
    
    # --- ETAPA 5: Calcular Métricas de Componente (Grafo Não-Direcionado) ---
    print("\n--- ETAPA 5: Calculando Métricas de Componente (Grafo Não-Direcionado) ---")

    # 5.1. Coeficiente de Clusterização
    print("  - Calculando Coeficiente de Clusterização Médio...")
    start_time = time.time()
    metricas["Coeficiente de Clusterização Médio"] = nx.average_clustering(G_nao_direcionado)
    print(f"    -> Concluído em {time.time() - start_time:.2f}s")
    
    # 5.2. Componente Gigante 
    print("  - Identificando Componente Gigante...")
    G_giant = nx.Graph()
    try:
        if G_nao_direcionado.number_of_nodes() > 0:
            componentes = list(nx.connected_components(G_nao_direcionado))
            if componentes:
                giant_component_nodes = max(componentes, key=len)
                G_giant = G_nao_direcionado.subgraph(giant_component_nodes).copy()
            else:
                giant_component_nodes = set()
        else:
            giant_component_nodes = set()
    except Exception as e:
        print(f"    -> ERRO ao identificar componente gigante: {e}")
        giant_component_nodes = set()

    metricas["Tamanho do Componente Gigante"] = G_giant.number_of_nodes()
    print(f"    -> Componente Gigante encontrado com {metricas['Tamanho do Componente Gigante']} nós.")
    
    # 5.3. Nós Isolados
    # Definição: Todos os nós que NÃO estão no componente gigante.
    num_isolados = G_direcionado.number_of_nodes() - G_giant.number_of_nodes()
    metricas["Quantidade de Nós isolados"] = num_isolados
    print(f"    -> {num_isolados} nós 'isolados' (fora do comp. gigante).")

    # 5.4. Métricas do Componente Gigante (Caminho Médio, Diâmetro)
    if G_giant.number_of_nodes() > 1:
        print("  - Calculando Comprimento Médio do Caminho (pode demorar)...")
        start_time = time.time()
        try:
            metricas["Comprimento Médio do Caminho"] = nx.average_shortest_path_length(G_giant)
            print(f"    -> Concluído em {time.time() - start_time:.2f}s")
        except nx.NetworkXError as e:
            print(f"    -> AVISO: Não foi possível calcular ({e}). Marcando como 'inf'.")
            metricas["Comprimento Médio do Caminho"] = float('inf')

        print("  - Calculando Diâmetro da Rede (pode demorar)...")
        start_time = time.time()
        try:
            metricas["Diâmetro da Rede"] = nx.diameter(G_giant)
            print(f"    -> Concluído em {time.time() - start_time:.2f}s")
        except nx.NetworkXError as e:
            print(f"    -> AVISO: Não foi possível calcular ({e}). Marcando como 'inf'.")
            metricas["Diâmetro da Rede"] = float('inf')
    else:
        print("  - PICO: Componente gigante muito pequeno para Diâmetro/Caminho Médio.")
        metricas["Comprimento Médio do Caminho"] = 0.0
        metricas["Diâmetro da Rede"] = 0

    # 5.5. Detecção de Comunidades
    print("  - Calculando Comunidades (Louvain)...")
    try:
        partition = community_louvain.best_partition(
            G_giant, 
            resolution=LOUVAIN_RESOLUTION, 
            random_state=RANDOM_SEED
        )
        num_communities = len(set(partition.values()))
        metricas["Quantidade de Comunidades Temáticas"] = num_communities
        print(f"    -> {num_communities} comunidades detectadas.")
    except Exception as e:
        print(f"    -> ERRO ao detectar comunidades: {e}")
        metricas["Quantidade de Comunidades Temáticas"] = 0

    # --- ETAPA 6: Imprimir Resultados Finais ---
    print("\n\n--- MÉTRICAS GLOBAIS FINAIS (Metodologia Aderente) ---")
    print("=" * 55)
    
    for chave, valor in metricas.items():
        if isinstance(valor, float):
            print(f"{chave:<38}: {valor:.6f}")
        else:
            print(f"{chave:<38}: {valor}")
            
    print("=" * 55)
    print("Cálculo de métricas globais concluído.")


# --- BLOCO DE EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    start_total_time = time.time()
    calcular_metricas_globais_aderentes()
    end_total_time = time.time()
    print(f"\nTempo total de execução: {end_total_time - start_total_time:.2f} segundos.")

✅ Configurações e funções do notebook carregadas.


--- ETAPA 1: Carregando Dados Brutos ---
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Carregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.

--- ETAPA 2: Calculando Métricas Não-Grafo ---
  - 1624 categorias únicas encontradas.
  - 851 editores únicos encontrados.

--- ETAPA 3: Construindo Grafos (Metodologia Célula 2) ---
Construindo grafo direcionado com resolução ITERATIVA de redirects...
  - Filtrando verbetes reais e criando mapa Título->ID...
  - 3294 verbetes reais identificados.
  - Adicionando nós ao grafo...
  - 3294 nós adicionados.
  - Adicionando arestas (resolvendo cadeias de redirecionamentos)...
Construção do grafo concluída:
  - Nós (Verbetes Reais): 3294
  - Arestas Adicionadas (tentativas): 16071
  - Arestas Únicas no Grafo: 15979
  - Referências que passaram por resolução (1+ sal

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Any, Optional
import time
import sys
import math

sys.setrecursionlimit(5000)

# --- 1. SEÇÃO DE CONFIGURAÇÃO CENTRALIZADA  ---
INPUT_DIR = Path('../dados/dados_com_flags_redirecionamento') 

FINAL_DATA_FILENAME = 'dados_com_constraint_novo.json'

print("✅ Configurações e funções do notebook carregadas.\n")

# --- 2. FUNÇÃO AUXILIAR ---

def carregar_dados_json(filepath: Path) -> Optional[List[Dict[str, Any]]]:
    """
    Carrega dados de um arquivo JSON e retorna especificamente a lista
    armazenada sob a chave 'verbetes_completo'.
    (Função da Célula 1)
    """
    if not filepath.exists():
        print(f"ERRO: Arquivo não encontrado em '{filepath}'")
        return None 
    print(f"Carregando dados de '{filepath}'...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            dados_completos = json.load(f)
        lista_verbetes = dados_completos.get('verbetes_completo', [])
        if not isinstance(lista_verbetes, list):
             print(f"ERRO: O conteúdo da chave 'verbetes_completo' em '{filepath}' não é uma lista.")
             return None 
        if not lista_verbetes:
             print(f"AVISO: A lista 'verbetes_completo' em '{filepath}' está vazia.")
        return lista_verbetes 
    except json.JSONDecodeError as e:
        print(f"ERRO: Falha ao decodificar o JSON em '{filepath}': {e}")
        return None
    except Exception as e:
        print(f"Erro inesperado ao carregar JSON em {filepath}: {e}")
        return None

# --- 3. FUNÇÃO PRINCIPAL: GERAR HISTOGRAMA  ---

def gerar_histograma_graus_precalculado():
    """
    Função principal para carregar os dados JÁ PROCESSADOS, 
    extrair os graus e gerar os histogramas (linear e log-log).
    """
    
    # --- ETAPA 1: Carregar Dados Pré-calculados ---
    print("\n--- ETAPA 1: Carregando Dados Pré-calculados ---")
    input_path = INPUT_DIR / FINAL_DATA_FILENAME

    verbetes_com_metricas = carregar_dados_json(input_path)
    if not verbetes_com_metricas:
        print(f"ERRO: Falha ao carregar dados de '{input_path}'. Abortando.")
        return
    
    print(f"Dados de {len(verbetes_com_metricas)} verbetes carregados.")

    # --- ETAPA 2: Extrair Graus do JSON e Colocar no Pandas ---
    print("\n--- ETAPA 2: Extraindo Graus Totais (Pré-calculados) ---")
    
    # Extrai o 'total_degree' de cada verbete na lista
    try:
        degrees_list = [
            v['total_degree'] 
            for v in verbetes_com_metricas 
            if 'total_degree' in v and isinstance(v['total_degree'], (int, float))
        ]
    except KeyError:
        print("ERRO: A chave 'total_degree' não foi encontrada em um ou mais verbetes.")
        print("Certifique-se de que o JSON de entrada é o resultado da Célula 2 (ou posterior).")
        return
    
    # Coloca em um DataFrame do Pandas para facilitar a análise
    df_degrees = pd.DataFrame(degrees_list, columns=['total_degree'])
    
    print(f"Graus extraídos para {len(df_degrees)} nós.")
    print(f"  - Grau Médio: {df_degrees['total_degree'].mean():.2f}")
    print(f"  - Grau Máximo: {df_degrees['total_degree'].max()}")
    print(f"  - Grau Mínimo: {df_degrees['total_degree'].min()}")
    print(f"  - Mediana do Grau: {df_degrees['total_degree'].median()}")
    
    grau_zero_ou_um = len(df_degrees[df_degrees['total_degree'] <= 1])
    print(f"  - Nós com Grau 0 ou 1: {grau_zero_ou_um} (Esperado ~297 isolados + órfãos)")

    # --- ETAPA 3: Plot 1 - Histograma Linear (Escala Padrão) ---
    print("\n--- ETAPA 3: Gerando Histograma em Escala Linear ---")
    
    max_degree = df_degrees['total_degree'].max()

    bins = min(int(max_degree) + 1, 100) 
    
    plt.figure(figsize=(12, 7))
    plt.hist(df_degrees['total_degree'], bins=bins, edgecolor='black', alpha=0.7)
    plt.title('Distribuição de Graus Totais (Escala Linear)', fontsize=16)
    plt.xlabel('Grau Total (Total Degree)', fontsize=12)
    plt.ylabel('Frequência (Contagem de Nós)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    output_filename_linear = 'histograma_graus_linear.png'
    plt.savefig(output_filename_linear, dpi=150, bbox_inches='tight')
    plt.close() 
    
    print(f"  -> Imagem salva como '{output_filename_linear}'")
    print("     (Este gráfico provavelmente mostrará tudo espremido à esquerda. Isso é esperado.)")


    # --- ETAPA 4: Plot 2 - Gráfico de Distribuição (Escala Log-Log) ---
    print("\n--- ETAPA 4: Gerando Gráfico de Distribuição (Escala Log-Log) ---")
    print("     (Este é o gráfico analiticamente mais importante para redes scale-free)")

    # 1. Contar a frequência de cada valor de grau
    degree_counts = df_degrees['total_degree'].value_counts().sort_index()
    
    # 2. Criar um novo DataFrame com [grau, contagem]
    df_counts = pd.DataFrame({
        'degree': degree_counts.index, 
        'count': degree_counts.values
    })
    
    # 3. Filtrar grau 0, pois log(0) é indefinido e não entra no gráfico log
    df_counts_plot = df_counts[df_counts['degree'] > 0]

    plt.figure(figsize=(12, 7))
    plt.scatter(df_counts_plot['degree'], df_counts_plot['count'], alpha=0.7, s=30) 
    
    plt.xscale('log')
    plt.yscale('log')
    
    plt.title('Distribuição de Graus Totais (Escala Log-Log)', fontsize=16)
    plt.xlabel('Grau Total (k) - Escala Log', fontsize=12)
    plt.ylabel('Contagem de Nós P(k) - Escala Log', fontsize=12)
    plt.grid(True, which="both", ls="--", alpha=0.5) 

    output_filename_loglog = 'histograma_graus_log_log.png'
    plt.savefig(output_filename_loglog, dpi=150, bbox_inches='tight')
    plt.close() 

    print(f"  -> Imagem salva como '{output_filename_loglog}'")
    print("     (Procure por uma tendência de linha reta decrescente, a 'assinatura' da Lei de Potência)")


# --- BLOCO DE EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    start_total_time = time.time()
    gerar_histograma_graus_precalculado()
    end_total_time = time.time()
    print(f"\nTempo total de execução: {end_total_time - start_total_time:.2f} segundos.")

✅ Configurações e funções do notebook carregadas.


--- ETAPA 1: Carregando Dados Pré-calculados ---
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_constraint_novo.json'...
Dados de 3294 verbetes carregados.

--- ETAPA 2: Extraindo Graus Totais (Pré-calculados) ---
Graus extraídos para 3294 nós.
  - Grau Médio: 9.70
  - Grau Máximo: 435
  - Grau Mínimo: 0
  - Mediana do Grau: 7.0
  - Nós com Grau 0 ou 1: 456 (Esperado ~297 isolados + órfãos)

--- ETAPA 3: Gerando Histograma em Escala Linear ---
  -> Imagem salva como 'histograma_graus_linear.png'
     (Este gráfico provavelmente mostrará tudo espremido à esquerda. Isso é esperado.)

--- ETAPA 4: Gerando Gráfico de Distribuição (Escala Log-Log) ---
     (Este é o gráfico analiticamente mais importante para redes scale-free)
  -> Imagem salva como 'histograma_graus_log_log.png'
     (Procure por uma tendência de linha reta decrescente, a 'assinatura' da Lei de Potência)

Tempo total de execução: 1.20 segundos.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Any, Optional
import time
import sys
import math

sys.setrecursionlimit(5000)

# --- 1. SEÇÃO DE CONFIGURAÇÃO CENTRALIZADA  ---
INPUT_DIR = Path('../dados/dados_com_flags_redirecionamento') 

FINAL_DATA_FILENAME = 'dados_com_constraint_novo.json'

print("✅ Configurações e funções do notebook carregadas.\n")

# --- 2. FUNÇÃO AUXILIAR  ---

def carregar_dados_json(filepath: Path) -> Optional[List[Dict[str, Any]]]:
    """
    Carrega dados de um arquivo JSON e retorna especificamente a lista
    armazenada sob a chave 'verbetes_completo'.
    (Função da Célula 1)
    """
    if not filepath.exists():
        print(f"ERRO: Arquivo não encontrado em '{filepath}'")
        return None 
    print(f"Carregando dados de '{filepath}'...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            dados_completos = json.load(f)
        lista_verbetes = dados_completos.get('verbetes_completo', [])
        if not isinstance(lista_verbetes, list):
             print(f"ERRO: O conteúdo da chave 'verbetes_completo' em '{filepath}' não é uma lista.")
             return None 
        if not lista_verbetes:
             print(f"AVISO: A lista 'verbetes_completo' em '{filepath}' está vazia.")
        return lista_verbetes 
    except json.JSONDecodeError as e:
        print(f"ERRO: Falha ao decodificar o JSON em '{filepath}': {e}")
        return None
    except Exception as e:
        print(f"Erro inesperado ao carregar JSON em {filepath}: {e}")
        return None

# ---  GERAR NOVOS HISTOGRAMAS ---

def gerar_histogramas_cluster_betweenness():
    """
    Função principal para carregar os dados JÁ PROCESSADOS, 
    extrair as métricas de clusterização e intermediação,
    e gerar os histogramas.
    """
    
    # --- ETAPA 1: Carregar Dados Pré-calculados ---
    print("\n--- ETAPA 1: Carregando Dados Pré-calculados ---")
    input_path = INPUT_DIR / FINAL_DATA_FILENAME

    verbetes_com_metricas = carregar_dados_json(input_path)
    if not verbetes_com_metricas:
        print(f"ERRO: Falha ao carregar dados de '{input_path}'. Abortando.")
        return
    
    print(f"Dados de {len(verbetes_com_metricas)} verbetes carregados.")

    # --- ETAPA 2: Colocar dados no Pandas ---
    print("\n--- ETAPA 2: Extraindo Métricas (Pré-calculadas) ---")
    
    df_metricas = pd.DataFrame(verbetes_com_metricas)
    
    # Verifica se as colunas necessárias existem
    colunas_necessarias = ['clustering_coefficient', 'betweenness_centrality']
    if not all(col in df_metricas.columns for col in colunas_necessarias):
        print("ERRO: O JSON de entrada não contém as colunas necessárias.")
        print(f"Verifique se '{colunas_necessarias[0]}' e '{colunas_necessarias[1]}' existem.")
        return

    # --- ETAPA 3: Plot 1 - Histograma do Coeficiente de Clusterização Local ---
    print("\n--- ETAPA 3: Gerando Histograma de Clusterização Local ---")
    
    metric_cluster = 'clustering_coefficient'
    
    # Conta quantos nós têm clusterização 0
    cluster_zero_count = len(df_metricas[df_metricas[metric_cluster] == 0])
    print(f"  - Nós com Clusterização Zero: {cluster_zero_count} (de {len(df_metricas)})")
    print(f"  - Média da Clusterização: {df_metricas[metric_cluster].mean():.4f} (Consistente com 0.2095)")
    print(f"  - Mediana da Clusterização: {df_metricas[metric_cluster].median():.4f}")

    bins_cluster = 50 
    
    plt.figure(figsize=(12, 7))
    plt.hist(df_metricas[metric_cluster], bins=bins_cluster, edgecolor='black', alpha=0.7, color='blue')
    plt.title('Distribuição do Coeficiente de Clusterização Local', fontsize=16)
    plt.xlabel('Coeficiente de Clusterização (Local)', fontsize=12)
    plt.ylabel('Frequência (Contagem de Nós)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    output_filename_cluster = 'histograma_clusterizacao_local.png'
    plt.savefig(output_filename_cluster, dpi=150, bbox_inches='tight')
    plt.close() 
    
    print(f"  -> Imagem salva como '{output_filename_cluster}'")
    print("     (Procure por um pico em '0.0' e por outros picos em valores altos)")

    # --- ETAPA 4: Plot 2 - Histograma da Centralidade de Intermediação (Linear) ---
    print("\n--- ETAPA 4: Gerando Histograma de Intermediação (Escala Linear) ---")
    
    metric_between = 'betweenness_centrality'
    
    # Conta quantos nós têm intermediação 0 
    between_zero_count = len(df_metricas[df_metricas[metric_between] == 0])
    print(f"  - Nós com Intermediação Zero: {between_zero_count} (de {len(df_metricas)})")
    print(f"  - Média da Intermediação: {df_metricas[metric_between].mean():.4f}")
    print(f"  - Máximo de Intermediação: {df_metricas[metric_between].max():.4f}")

    bins_between = 50
    
    plt.figure(figsize=(12, 7))
    plt.hist(df_metricas[metric_between], bins=bins_between, edgecolor='black', alpha=0.7, color='green')
    plt.title('Distribuição da Centralidade de Intermediação (Escala Linear)', fontsize=16)
    plt.xlabel('Centralidade de Intermediação (Betweenness)', fontsize=12)
    plt.ylabel('Frequência (Contagem de Nós)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    output_filename_b_linear = 'histograma_betweenness_linear.png'
    plt.savefig(output_filename_b_linear, dpi=150, bbox_inches='tight')
    plt.close() 

    print(f"  -> Imagem salva como '{output_filename_b_linear}'")
    print("     (Este gráfico DEVE parecer 'esmagado' à esquerda. Isso é esperado.)")


    # --- ETAPA 5: Plot 3 - Histograma da Centralidade de Intermediação ---
    print("\n--- ETAPA 5: Gerando Histograma de Intermediação (Escala Log na Contagem) ---")

    plt.figure(figsize=(12, 7))
    
    plt.hist(df_metricas[metric_between], bins=bins_between, edgecolor='black', alpha=0.7, color='green')
    
    plt.yscale('log')
    
    plt.title('Distribuição da Centralidade de Intermediação (Contagem em Escala Log)', fontsize=16)
    plt.xlabel('Centralidade de Intermediação (Betweenness)', fontsize=12)
    plt.ylabel('Frequência (Contagem de Nós) - Escala Log', fontsize=12)
    plt.grid(True, which="both", ls="--", alpha=0.5) 

    output_filename_b_log_y = 'histograma_betweenness_log-count.png'
    plt.savefig(output_filename_b_log_y, dpi=150, bbox_inches='tight')
    plt.close() 

    print(f"  -> Imagem salva como '{output_filename_b_log_y}'")
    print("     (Este gráfico permite ver a 'cauda longa' dos nós corretores/pontes)")


# --- BLOCO DE EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    start_total_time = time.time()
    gerar_histogramas_cluster_betweenness()
    end_total_time = time.time()
    print(f"\nTempo total de execução: {end_total_time - start_total_time:.2f} segundos.")

✅ Configurações e funções do notebook carregadas.


--- ETAPA 1: Carregando Dados Pré-calculados ---
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_constraint_novo.json'...
Dados de 3294 verbetes carregados.

--- ETAPA 2: Extraindo Métricas (Pré-calculadas) ---

--- ETAPA 3: Gerando Histograma de Clusterização Local ---
  - Nós com Clusterização Zero: 895 (de 3294)
  - Média da Clusterização: 0.2095 (Consistente com 0.2095)
  - Mediana da Clusterização: 0.1429
  -> Imagem salva como 'histograma_clusterizacao_local.png'
     (Procure por um pico em '0.0' e por outros picos em valores altos)

--- ETAPA 4: Gerando Histograma de Intermediação (Escala Linear) ---
  - Nós com Intermediação Zero: 1170 (de 3294)
  - Média da Intermediação: 0.0007
  - Máximo de Intermediação: 0.1928
  -> Imagem salva como 'histograma_betweenness_linear.png'
     (Este gráfico DEVE parecer 'esmagado' à esquerda. Isso é esperado.)

--- ETAPA 5: Gerando Histograma de Intermediação (Escala